# **Join Query Example**

- All names have been cleansed for anonymity.
- All SQL observed was written by myself.

In [ ]:
/*------ XXXXX Meta & Google Ads Join Query ----*/

-- Author: Quinn Olynyk --
-- Date: xxxx --
-- Goal: Join data sources: Google Ads, Meta, for client: XXXX--

CREATE OR REPLACE TABLE `XXX` AS

/* --- Google Ads aggregated to date --- */
WITH g_ads AS (
  SELECT
    CAST(Date AS DATE) AS date,
    ROUND(SUM(CAST(cost__xxx AS FLOAT64)), 2) AS ga_cost,
    SUM(CAST(Clicks_xxx AS INT64)) AS ga_clicks,
    SUM(CAST(Impressions_xxx AS INT64)) AS ga_impressions,
    SUM(CAST(Conversions__Google_Ads AS INT64)) AS ga_conversions,
    ROUND(SUM(CAST(Conv__Value__Google_Ads AS FLOAT64)), 2) AS ga_revenue
  FROM `xxxx`
  WHERE Google_Ads_Funnel_Export_Type = "SUM"
  GROUP BY 1
),

/* --- Meta aggregated to date --- */
meta AS (
  SELECT
    CAST(Date AS DATE) AS date,
    ROUND(SUM(CAST(Amount_Spent__Facebook_Ads AS FLOAT64)), 2) AS meta_cost,
    SUM(CAST(Clicks AS INT64)) AS meta_clicks,
    SUM(CAST(Impressions AS INT64)) AS meta_impressions,
    ROUND(SUM(CAST(Purchases_Conversion_Value__Facebook_Ads AS FLOAT64)), 2) AS meta_revenue
  FROM `xxxxx`
  GROUP BY 1
)

/* --- Join at matching grain: date --- */
SELECT
  COALESCE(g_ads.date, meta.date) AS date,
  COALESCE(ga_cost, 0) AS ga_cost,
  COALESCE(ga_clicks, 0) AS ga_clicks,
  COALESCE(ga_impressions, 0) AS ga_impressions,
  COALESCE(ga_conversions, 0) AS ga_conversions,
  COALESCE(ga_revenue, 0) AS ga_revenue,
  COALESCE(meta_cost, 0) AS meta_cost,
  COALESCE(meta_clicks, 0) AS meta_clicks,
  COALESCE(meta_impressions, 0) AS meta_impressions,
  COALESCE(meta_revenue, 0) AS meta_revenue
FROM g_ads
FULL OUTER JOIN meta
  ON g_ads.date = meta.date
WHERE COALESCE(g_ads.date, meta.date) >= '2026-03-01'
ORDER BY date;